# BOOMER-PY CLI Tutorial: Command-Line Probabilistic Reasoning

This tutorial demonstrates how to use BOOMER-PY from the command line, working with YAML knowledge bases and various output formats.

## Table of Contents
1. [Setup and Basic CLI Usage](#setup)
2. [Creating YAML Knowledge Bases](#yaml-kb)
3. [Solving Knowledge Bases](#solve)
4. [Converting Between Formats](#convert)
5. [Merging Knowledge Bases](#merge)
6. [Output Formats](#output)
7. [Advanced Options](#advanced)
8. [Real-World Example](#realworld)

## 1. Setup and Basic CLI Usage {#setup}

BOOMER-PY provides a rich CLI for probabilistic reasoning without writing Python code.

First, let's check the CLI is working and see available commands:

In [1]:
%%bash
# Check BOOMER-PY CLI is available
uv run python -m boomer.cli --help | head -20

Usage: python -m boomer.cli [OPTIONS] COMMAND [ARGS]...

  BOOMER - Bayesian OWL Ontology MErgER

Op

tions:
  -h, --help  Show this message and exit.

Commands:
  convert        Convert between differe

nt KB formats.
  eval           Evaluate predicted facts (from a BOOMER Solution)...
  extract      

  Extract a sub-KB from a KB using a file containing...
  grid-search    Perform a grid search over 

search configurations...
  list-datasets  List all available built-in datasets.
  merge          Mer

ge multiple KB files into a single KB.
  solve          BOOMER - Bayesian OWL Ontology MErgER


## 2. Creating YAML Knowledge Bases {#yaml-kb}

YAML is a human-readable format perfect for defining knowledge bases. Let's create a simple animal classification KB:

In [2]:
%%bash
# Create a simple YAML knowledge base
cat > /tmp/animals.yaml << 'EOF'
name: Animal Classification
description: Mapping common names to scientific classifications

# Probabilistic facts (pfacts)
pfacts:
  # High confidence mappings
  - fact:
      fact_type: EquivalentTo
      sub: cat
      equivalent: Felis_catus
    prob: 0.95
    
  - fact:
      fact_type: EquivalentTo
      sub: dog
      equivalent: Canis_familiaris
    prob: 0.95
    
  # Potential confusion - low probability
  - fact:
      fact_type: EquivalentTo
      sub: cat
      equivalent: Canis_familiaris
    prob: 0.05
    
  # Hierarchical relationships
  - fact:
      fact_type: SubClassOf
      sub: cat
      sup: mammal
    prob: 0.99
    
  - fact:
      fact_type: SubClassOf
      sub: dog
      sup: mammal
    prob: 0.99
EOF

echo "Created /tmp/animals.yaml"
echo "---"
echo "First 10 lines:"
head -10 /tmp/animals.yaml

Created /tmp/animals.yaml
---
First 10 lines:


name: Animal Classification
description: Mapping common names to scientific classifications

# Proba

bilistic facts (pfacts)
pfacts:
  # High confidence mappings
  - fact:
      fact_type: EquivalentTo


      sub: cat
      equivalent: Felis_catus


## 3. Solving Knowledge Bases {#solve}

The `solve` command (default) performs probabilistic reasoning to find the most consistent interpretation:

In [3]:
%%bash
# Solve the knowledge base with default settings
uv run python -m boomer.cli solve --timeout 30 /tmp/animals.yaml

Loaded KB with 0 facts and 5 probabilistic facts
Starting search...


Solving KB: Animal Classification with 5 pfacts; threshold=200
Search completed in 0.0170 seconds
Kn

owledge Base: Animal Classification
KB num pfacts: 5
Description: Mapping common names to scientific

 classifications

Search Statistics:
  Confidence: 0.9500
  Prior Probability: 8.4031e-01
  Posterio

r Probability: 0.7694
  Combinations Explored: 37
  Satisfiable Combinations: 32
  Time Elapsed: 0.0

...

sub='dog' equivalent='Canis_familiaris' :: prior: 0.95 posterior: 0.9542195392837551
 * False fact_t

ype='EquivalentTo' sub='cat' equivalent='Canis_familiaris' :: prior: 0.05 posterior: 0.1301712463913

552
 * True fact_type='SubClassOf' sub='cat' sup='mammal' :: prior: 0.99 posterior: 0.99127447308978

68
 * True fact_type='SubClassOf' sub='dog' sup='mammal' :: prior: 0.99 posterior: 0.991274473089786

8



In [4]:
%%bash
# Solve with markdown output for better readability
uv run python -m boomer.cli solve --timeout 30 /tmp/animals.yaml -O markdown

Loaded KB with 0 facts and 5 probabilistic facts
Starting search...


Solving KB: Animal Classification with 5 pfacts; threshold=200
Search completed in 0.0178 seconds
Kn

owledge Base: Animal Classification
KB num pfacts: 5
Description: Mapping common names to scientific

 classifications

Search Statistics:
  Confidence: 0.9500
  Prior Probability: 8.4031e-01
  Posterio

r Probability: 0.7694
  Combinations Explored: 38
  Satisfiable Combinations: 32
  Time Elapsed: 0.0

...

 0.95 posterior: 0.9499999999999998
 * True fact_type='EquivalentTo' sub='dog' equivalent='Canis_fam

iliaris' :: prior: 0.95 posterior: 0.9542195392837551
 * False fact_type='EquivalentTo' sub='cat' eq

uivalent='Canis_familiaris' :: prior: 0.05 posterior: 0.1301712463913552
 * True fact_type='SubClass

Of' sub='cat' sup='mammal' :: prior: 0.99 posterior: 0.9912744730897868
 * True fact_type='SubClassO

f' sub='dog' sup='mammal' :: prior: 0.99 posterior: 0.9912744730897868



## 4. Converting Between Formats {#convert}

BOOMER-PY can convert between different KB formats. Let's create a more complex KB and convert it:

In [5]:
%%bash
# Create a taxonomy alignment KB
cat > /tmp/taxonomy.yaml << 'EOF'
name: Taxonomy Alignment
description: Aligning two classification systems
comments: Example of ontology matching between System A and System B

pfacts:
  # System A to System B mappings
  - fact: {fact_type: EquivalentTo, sub: A1, equivalent: B1}
    prob: 0.8
  - fact: {fact_type: EquivalentTo, sub: A1, equivalent: B2}  
    prob: 0.3
  - fact: {fact_type: EquivalentTo, sub: A2, equivalent: B2}
    prob: 0.9
  - fact: {fact_type: EquivalentTo, sub: A2, equivalent: B3}
    prob: 0.2
    
  # Hierarchies within systems
  - fact: {fact_type: SubClassOf, sub: A1, sup: A_root}
    prob: 0.99
  - fact: {fact_type: SubClassOf, sub: A2, sup: A_root}
    prob: 0.99
  - fact: {fact_type: SubClassOf, sub: B1, sup: B_root}
    prob: 0.99
  - fact: {fact_type: SubClassOf, sub: B2, sup: B_root}
    prob: 0.99
    
# Labels for better readability
labels:
  A1: "Category Alpha-1"
  A2: "Category Alpha-2"
  B1: "Type Beta-1"
  B2: "Type Beta-2"
EOF

echo "Created taxonomy.yaml"

Created taxonomy.yaml


In [6]:
%%bash
# Convert YAML to JSON
uv run python -m boomer.cli convert /tmp/taxonomy.yaml -o /tmp/taxonomy.json
echo "---"
echo "JSON output (first 20 lines):"
head -20 /tmp/taxonomy.json

Converted /tmp/taxonomy.yaml (yaml) to /tmp/taxonomy.json (json)


---
JSON output (first 20 lines):


{
  "facts": [],
  "pfacts": [
    {
      "fact": {
        "fact_type": "EquivalentTo",
        "s

ub": "A1",
        "equivalent": "B1"
      },
      "prob": 0.8
    },
    {
      "fact": {
      

  "fact_type": "EquivalentTo",
        "sub": "A1",
        "equivalent": "B2"
      },
      "prob"

: 0.3
    },
    {


In [7]:
%%bash
# Convert one of the built-in datasets to YAML
uv run python -m boomer.cli convert boomer.datasets.animals -o /tmp/animals_dataset.yaml
echo "Converted built-in animals dataset to YAML"
echo "---"
echo "First 15 lines:"
head -15 /tmp/animals_dataset.yaml

Converted boomer.datasets.animals (py) to /tmp/animals_dataset.yaml (yaml)


Converted built-in animals dataset to YAML
---
First 15 lines:


name: Animals
description: An ontology alignment example between common animal names and scientific


  taxonomy
comments: Maps between common terms (cat, dog, furry_animal) and scientific terms
  (Feli

x, Canus, Mammalia)
facts:
- fact_type: ProperSubClassOf
  sub: Felix
  sup: Mammalia
- fact_type: P

roperSubClassOf
  sub: Canus
  sup: Mammalia
- fact_type: MemberOfDisjointGroup
  sub: cat
  group: 

Common


%%bash
# Solve with custom search configuration
echo "=== With limited iterations (fast but potentially suboptimal) ==="
uv run python -m boomer.cli solve /tmp/complex.yaml \
  --max-iterations 100 \
  --timeout 5 \
  -O markdown | grep -A5 "Solution Statistics"

In [8]:
%%bash
# Create another KB to merge
cat > /tmp/plants.yaml << 'EOF'
name: Plant Classification
description: Basic plant taxonomy

pfacts:
  - fact: {fact_type: EquivalentTo, sub: oak, equivalent: Quercus}
    prob: 0.95
  - fact: {fact_type: EquivalentTo, sub: rose, equivalent: Rosa}
    prob: 0.95
  - fact: {fact_type: SubClassOf, sub: oak, sup: tree}
    prob: 0.99
  - fact: {fact_type: SubClassOf, sub: rose, sup: flower}
    prob: 0.99
  - fact: {fact_type: SubClassOf, sub: tree, sup: plant}
    prob: 0.99
  - fact: {fact_type: SubClassOf, sub: flower, sup: plant}
    prob: 0.99
EOF

# Merge animals and plants KBs
uv run python -m boomer.cli merge /tmp/animals.yaml /tmp/plants.yaml \
  -o /tmp/merged.yaml \
  -n "Living Things" \
  -D "Combined animal and plant classifications"

echo "Merged KB created"
echo "---"
echo "Number of facts in merged KB:"
grep -c "prob:" /tmp/merged.yaml

Merged 2 files into /tmp/merged.yaml (yaml)
Final KB contains 0 facts and 11 probabilistic facts


Merged KB created
---
Number of facts in merged KB:


11


In [9]:
%%bash
# Solve the merged KB
uv run python -m boomer.cli solve --timeout 30 /tmp/merged.yaml -O markdown | head -30

Loaded KB with 0 facts and 11 probabilistic facts
Starting search...
Solving KB: Living Things with 

11 pfacts; threshold=200
Search timeout after 30.00 seconds
Search completed in 30.0461 seconds
Know

ledge Base: Living Things
KB num pfacts: 11
Description: Combined animal and plant classifications



Search Statistics:
  Confidence: 0.9500
  Prior Probability: 7.2850e-01
  Posterior Probability: 0.6

672
  Combinations Explored: 1224
  Satisfiable Combinations: 1177
  Time Elapsed: 30.0387 seconds
 

...

p='plant' (posterior: 0.9900)
  fact_type='SubClassOf' sub='flower' sup='plant' (posterior: 0.9901)


  fact_type='EquivalentTo' sub='cat' equivalent='Felis_catus' (posterior: 0.9502)
  fact_type='Equiv

alentTo' sub='dog' equivalent='Canis_familiaris' (posterior: 0.9542)
  fact_type='EquivalentTo' sub=

'oak' equivalent='Quercus' (posterior: 0.9500)
  fact_type='EquivalentTo' sub='rose' equivalent='Ros

a' (posterior: 0.9502)


## 6. Output Formats {#output}

BOOMER-PY supports multiple output formats for different use cases:

In [10]:
%%bash
# TSV output - useful for spreadsheets and further processing
uv run python -m boomer.cli solve --timeout 30 /tmp/animals.yaml -O tsv -o /tmp/solution.tsv --quiet
echo "TSV output saved to /tmp/solution.tsv"
echo "---"
echo "TSV contents:"
cat /tmp/solution.tsv

Solving KB: Animal Classification with 5 pfacts; threshold=200


TSV output saved to /tmp/solution.tsv
---
TSV contents:


# BOOMER Solution TSV Output
#
# Metadata:
#   generated_date: 2025-09-24T20:45:51.699478
#   combin

ations: 38
#   satisfiable_combinations: 32
#   confidence: 0.9499999999999998
#   prior_probability

: 0.8403132374999999
#   posterior_probability: 0.7693985431741789
#   time_elapsed_seconds: 0.01769

6142196655273
#   timed_out: False
#
# Format: fact_type followed by arguments, then truth_value and

 probabilities
#
fact_type	arg1	arg2	truth_value	prior_probability	posterior_probability
Equivalent

To	cat	Felis_catus	True	0.95	0.9499999999999998
EquivalentTo	dog	Canis_familiaris	True	0.95	0.95421

95392837551
EquivalentTo	cat	Canis_familiaris	False	0.05	0.1301712463913552
SubClassOf	cat	mammal	

True	0.99	0.9912744730897868
SubClassOf	dog	mammal	True	0.99	0.9912744730897868


In [11]:
%%bash
# JSON output - for programmatic processing
uv run python -m boomer.cli solve --timeout 30 /tmp/taxonomy.yaml -O json -o /tmp/solution.json --quiet
echo "JSON solution saved"
echo "---"
echo "Solution statistics from JSON:"
uv run python -c "
import json
with open('/tmp/solution.json') as f:
    sol = json.load(f)
    print(f'Confidence: {sol[\"confidence\"]:.3f}')
    print(f'Combinations explored: {sol[\"number_of_combinations\"]}')
    print(f'Satisfiable combinations: {sol[\"number_of_satisfiable_combinations\"]}')
    print(f'Time elapsed: {sol.get(\"time_elapsed\", 0):.3f}s')
"

Solving KB: Taxonomy Alignment with 8 pfacts; threshold=200


JSON solution saved
---
Solution statistics from JSON:


Confidence: 0.700
Combinations explored: 372
Satisfiable combinations: 256
Time elapsed: 1.332s


In [12]:
%%bash
# YAML output - human-readable solution format
uv run python -m boomer.cli solve --timeout 30 /tmp/animals.yaml -O yaml -o /tmp/solution.yaml --quiet
echo "YAML solution:"
echo "---"
head -25 /tmp/solution.yaml

Solving KB: Animal Classification with 5 pfacts; threshold=200


YAML solution:
---


name: null
number_of_combinations: 36
number_of_satisfiable_combinations: 32
number_of_combinations_

explored_including_implicit: 38
number_of_components: null
confidence: 0.9499999999999998
prior_prob

: 0.8403132374999999
posterior_prob: 0.7693985431741789
proportion_of_combinations_explored: 1.0
gro

und_pfacts: []
solved_pfacts:
- pfact:
    fact:
      fact_type: EquivalentTo
      sub: cat
      

equivalent: Felis_catus
    prob: 0.95
  truth_value: true
  posterior_prob: 0.9499999999999998
  me

tadata: null
- pfact:
    fact:
      fact_type: EquivalentTo
      sub: dog
      equivalent: Canis

_familiaris


### SSSOM TSV Output

[SSSOM](https://mapping-commons.github.io/sssom/) is the standard format for ontology mappings. BOOMER can export solutions directly as SSSOM TSV, making them consumable by mapping pipelines (e.g., OAK, sssom-py):

In [ ]:
%%bash
# SSSOM output - standard ontology mapping format
uv run python -m boomer.cli solve --timeout 30 /tmp/diseases.yaml -O sssom -o /tmp/solution.sssom.tsv --quiet
echo "SSSOM output saved to /tmp/solution.sssom.tsv"
echo "---"
cat /tmp/solution.sssom.tsv

### OBOGraphs JSON Output

[OBOGraphs](https://github.com/geneontology/obographs) is the standard graph exchange format for ontologies, used by OAK, Monarch, and the broader OBO community:

In [ ]:
%%bash
# OBOGraphs JSON output - standard ontology graph format
uv run python -m boomer.cli solve --timeout 30 /tmp/diseases.yaml -O obographs -o /tmp/solution.obographs.json --quiet
echo "OBOGraphs JSON output saved to /tmp/solution.obographs.json"
echo "---"
echo "First 30 lines:"
python -m json.tool /tmp/solution.obographs.json | head -30

## 7. Advanced Options {#advanced}

The CLI provides various options for controlling the search process:

In [13]:
%%bash
# Solve with custom search configuration
echo "=== With limited iterations (fast but potentially suboptimal) ==="
uv run python -m boomer.cli solve /tmp/complex.yaml \
  --max-iterations 100 \
  --timeout 5 \
  -O markdown | tail -15

=== With limited iterations (fast but potentially suboptimal) ===


 * 0.7 confidence
 * 0.005644800000000001 prior probability
 * 0.36612702366127026 posterior probabi

lity
 * 0.1227 seconds elapsed
Grounding:
 * True fact_type='EquivalentTo' sub='X1' equivalent='Y1' 

:: prior: 0.7 posterior: 0.5404732254047323
 * False fact_type='EquivalentTo' sub='X1' equivalent='Y

2' :: prior: 0.6 posterior: 0.2773972602739726
 * False fact_type='EquivalentTo' sub='X1' equivalent

...

ivalent='Y2' :: prior: 0.7 posterior: 0.5230386052303861
 * False fact_type='EquivalentTo' sub='X2' 

equivalent='Y3' :: prior: 0.4 posterior: 0.017434620174346202
 * False fact_type='EquivalentTo' sub=

'X3' equivalent='Y1' :: prior: 0.5 posterior: 0.025217932752179328
 * False fact_type='EquivalentTo'

 sub='X3' equivalent='Y2' :: prior: 0.4 posterior: 0.017434620174346202
 * True fact_type='Equivalen

tTo' sub='X3' equivalent='Y3' :: prior: 0.8 posterior: 0.9321295143212952



In [14]:
%%bash
# Solve with more thorough search
echo "=== With more iterations (slower but more thorough) ==="
uv run python -m boomer.cli solve /tmp/complex.yaml \
  --max-iterations 10000 \
  --timeout 20 \
  -O markdown | tail -15

=== With more iterations (slower but more thorough) ===


 * 0.7 confidence
 * 0.005644800000000001 prior probability
 * 0.2053251855085115 posterior probabil

ity
 * 1.1806 seconds elapsed
Grounding:
 * True fact_type='EquivalentTo' sub='X1' equivalent='Y1' :

: prior: 0.7 posterior: 0.4057616761239632
 * False fact_type='EquivalentTo' sub='X1' equivalent='Y2

' :: prior: 0.6 posterior: 0.20977738978611954
 * False fact_type='EquivalentTo' sub='X1' equivalent

...

ivalent='Y2' :: prior: 0.7 posterior: 0.432649498035792
 * False fact_type='EquivalentTo' sub='X2' e

quivalent='Y3' :: prior: 0.4 posterior: 0.060061108686163225
 * False fact_type='EquivalentTo' sub='

X3' equivalent='Y1' :: prior: 0.5 posterior: 0.09271060672195541
 * False fact_type='EquivalentTo' s

ub='X3' equivalent='Y2' :: prior: 0.4 posterior: 0.060061108686163225
 * True fact_type='EquivalentT

o' sub='X3' equivalent='Y3' :: prior: 0.8 posterior: 0.6170231340026187



In [15]:
%%bash
# Solve with more thorough search
echo "=== With more iterations (slower but more thorough) ==="
uv run python -m boomer.cli solve /tmp/complex.yaml \
  --max-iterations 10000 \
  --timeout 30 \
  -O markdown | tail -15

=== With more iterations (slower but more thorough) ===


 * 0.6999999999999998 confidence
 * 0.005644800000000001 prior probability
 * 0.2053251855085115 pos

terior probability
 * 1.3197 seconds elapsed
Grounding:
 * True fact_type='EquivalentTo' sub='X1' eq

uivalent='Y1' :: prior: 0.7 posterior: 0.4057616761239632
 * False fact_type='EquivalentTo' sub='X1'

 equivalent='Y2' :: prior: 0.6 posterior: 0.20977738978611954
 * False fact_type='EquivalentTo' sub=

...

o' sub='X2' equivalent='Y2' :: prior: 0.7 posterior: 0.432649498035792
 * False fact_type='Equivalen

tTo' sub='X2' equivalent='Y3' :: prior: 0.4 posterior: 0.060061108686163225
 * False fact_type='Equi

valentTo' sub='X3' equivalent='Y1' :: prior: 0.5 posterior: 0.09271060672195544
 * False fact_type='

EquivalentTo' sub='X3' equivalent='Y2' :: prior: 0.4 posterior: 0.060061108686163225
 * True fact_ty

pe='EquivalentTo' sub='X3' equivalent='Y3' :: prior: 0.8 posterior: 0.6170231340026187



## 8. Real-World Example: Disease Mapping {#realworld}

Let's create a realistic example mapping disease classifications between MONDO and ICD-10:

In [16]:
%%bash
# Create a disease mapping KB
cat > /tmp/diseases.yaml << 'EOF'
name: Disease Ontology Mapping
description: Mapping MONDO disease ontology to ICD-10 codes
comments: Example mappings for common diseases

pfacts:
  # High-confidence direct mappings
  - fact:
      fact_type: EquivalentTo
      sub: "MONDO:0005015"  # Diabetes mellitus
      equivalent: "ICD10:E11"
    prob: 0.95
    
  - fact:
      fact_type: EquivalentTo
      sub: "MONDO:0005267"  # Heart disease
      equivalent: "ICD10:I51"
    prob: 0.90
    
  - fact:
      fact_type: EquivalentTo
      sub: "MONDO:0002251"  # Pneumonia
      equivalent: "ICD10:J18"
    prob: 0.92
    
  # Ambiguous mappings - one MONDO could map to multiple ICD codes
  - fact:
      fact_type: EquivalentTo
      sub: "MONDO:0005249"  # Hypertension
      equivalent: "ICD10:I10"  # Essential hypertension
    prob: 0.70
    
  - fact:
      fact_type: EquivalentTo
      sub: "MONDO:0005249"  # Hypertension
      equivalent: "ICD10:I11"  # Hypertensive heart disease
    prob: 0.40
    
  # Hierarchical relationships
  - fact:
      fact_type: SubClassOf
      sub: "MONDO:0005015"
      sup: "MONDO:0005066"  # Metabolic disease
    prob: 0.99
    
  - fact:
      fact_type: SubClassOf
      sub: "MONDO:0005267"
      sup: "MONDO:0005267_parent"  # Cardiovascular disease
    prob: 0.99
    
  - fact:
      fact_type: SubClassOf
      sub: "ICD10:E11"
      sup: "ICD10:E00-E89"  # Endocrine chapter
    prob: 0.99
    
  - fact:
      fact_type: SubClassOf
      sub: "ICD10:I51"
      sup: "ICD10:I00-I99"  # Circulatory chapter
    prob: 0.99

labels:
  "MONDO:0005015": "Diabetes mellitus"
  "MONDO:0005267": "Heart disease"
  "MONDO:0002251": "Pneumonia"
  "MONDO:0005249": "Hypertension"
  "MONDO:0005066": "Metabolic disease"
  "ICD10:E11": "Type 2 diabetes mellitus"
  "ICD10:I51": "Complications of heart disease"
  "ICD10:J18": "Pneumonia, unspecified organism"
  "ICD10:I10": "Essential (primary) hypertension"
  "ICD10:I11": "Hypertensive heart disease"
EOF

echo "Created disease mapping KB"

Created disease mapping KB


In [17]:
%%bash
# Solve the disease mapping with detailed output
uv run python -m boomer.cli solve /tmp/diseases.yaml \
  --timeout 60 \
  -O markdown

Loaded KB with 0 facts and 9 probabilistic facts
Starting search...


Solving KB: Disease Ontology Mapping with 9 pfacts; threshold=200
Search completed in 13.3022 second

s


Knowledge Base: Disease Ontology Mapping
KB num pfacts: 9
Description: Mapping MONDO disease ontolog

y to ICD-10 codes

Search Statistics:
  Confidence: 0.6000
  Prior Probability: 3.1735e-01
  Posteri

...

ONDO:0005015 (Diabetes mellitus) ⊆ MONDO:0005066 (Metabolic disease) :: prior: 0.99 posterior: 0.9

899999999999999
 * True MONDO:0005267 (Heart disease) ⊆ MONDO:0005267_parent :: prior: 0.99 poster

ior: 0.9899999999999999
 * True ICD10:E11 (Type 2 diabetes mellitus) ⊆ ICD10:E00-E89 :: prior: 0.9

9 posterior: 0.9899999999999999
 * True ICD10:I51 (Complications of heart disease) ⊆ ICD10:I00-I99

 :: prior: 0.99 posterior: 0.9899999999999999



In [18]:
%%bash
# Export solution as TSV for integration with other tools
uv run python -m boomer.cli solve /tmp/diseases.yaml \
  --timeout 60 \
  -O tsv \
  -o /tmp/disease_mappings.tsv \
  --quiet

echo "Disease mappings exported to TSV:"
echo "---"
cat /tmp/disease_mappings.tsv

Solving KB: Disease Ontology Mapping with 9 pfacts; threshold=200


Disease mappings exported to TSV:
---


# BOOMER Solution TSV Output
#
# Metadata:
#   generated_date: 2025-09-24T20:46:22.542545
#   combin

ations: 512
#   satisfiable_combinations: 512
#   confidence: 0.6000000000000001
#   prior_probabili

ty: 0.31735402501572
#   posterior_probability: 0.31735402501571985
#   time_elapsed_seconds: 11.457

...

n	Hypertensive heart disease	False	0.4	0.39999999999999813
SubClassOf	MONDO:0005015	MONDO:0005066	D

iabetes mellitus	Metabolic disease	True	0.99	0.9899999999999999
SubClassOf	MONDO:0005267	MONDO:0005

267_parent	Heart disease		True	0.99	0.9899999999999999
SubClassOf	ICD10:E11	ICD10:E00-E89	Type 2 di

abetes mellitus		True	0.99	0.9899999999999999
SubClassOf	ICD10:I51	ICD10:I00-I99	Complications of h

eart disease		True	0.99	0.9899999999999999


In [19]:
%%bash
# Filter for only accepted equivalences using grep
echo "Accepted disease mappings (EquivalentTo with truth_value=True):"
echo "---"
grep "^EquivalentTo.*True" /tmp/disease_mappings.tsv | cut -f1-5

Accepted disease mappings (EquivalentTo with truth_value=True):
---


EquivalentTo	MONDO:0005015	ICD10:E11	Diabetes mellitus	Type 2 diabetes mellitus
EquivalentTo	MONDO:0

005267	ICD10:I51	Heart disease	Complications of heart disease
EquivalentTo	MONDO:0002251	ICD10:J18	P

neumonia	Pneumonia, unspecified organism
EquivalentTo	MONDO:0005249	ICD10:I10	Hypertension	Essential

 (primary) hypertension


## Summary

This tutorial covered the BOOMER-PY command-line interface:

1. **YAML Knowledge Bases**: Human-readable format for defining facts and probabilities
2. **Solving**: Finding the most probable consistent interpretation
3. **Format Conversion**: Converting between YAML, JSON, and other formats
4. **Merging**: Combining multiple knowledge bases
5. **Output Formats**: TSV, JSON, YAML, and Markdown outputs
6. **Advanced Options**: Controlling search parameters
7. **Real-World Application**: Disease ontology mapping

The CLI provides a powerful interface for probabilistic reasoning without writing code, making it accessible for:
- **Data integration pipelines**: Use in shell scripts and workflows
- **Ontology alignment tasks**: Map between different classification systems
- **Quick experiments**: Test reasoning scenarios without programming
- **Batch processing**: Process multiple KB files programmatically

For more details, see the main [BOOMER-PY documentation](../index.md).